# iWildCam Results Analysis Notebook

This notebook is designed for post-run analysis of `iwildcam_iro` experiments produced by the standard `iro train` / `iro eval` JSONL artifacts.

What this notebook gives you:
- robust JSONL loading from nested results folders
- run health checks (failed/ok, seed coverage)
- algorithm-level summary tables (mean/std over seeds)
- validation metric visualizations (accuracy and macro-F1)
- training dynamics over steps from `result.history`
- optional IRO alpha-preference sweep analysis (if `iro eval` records exist)
- CSV + figure export for reports


In [ ]:
# Setup: imports, plotting defaults, and display options
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn", "-q"])
    import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RNG = np.random.default_rng(0)


## Configuration

Update paths if needed. The loader scans recursively for `*.jsonl`.


In [ ]:
# Configuration
RESULTS_ROOT_CANDIDATES = [
    Path("/home/c01josh/CISPA-scratch/c01josh/iro/runs/iwildcam_long/results"),
    Path("/home/c01josh/CISPA-scratch/c01josh/iro/runs/iwildcam_full/results"),
    Path("/home/c01josh/CISPA-scratch/c01josh/iro/iro_exp/results"),
]

RESULTS_ROOT = next((p for p in RESULTS_ROOT_CANDIDATES if p.exists()), RESULTS_ROOT_CANDIDATES[0])
DATASET_FILTER = "iwildcam"  # set to None to load all datasets
OUT_DIR = Path("/home/c01josh/CISPA-scratch/c01josh/iro/collected_results/iwildcam_analysis")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"RESULTS_ROOT exists: {RESULTS_ROOT.exists()}")
print(f"OUT_DIR: {OUT_DIR}")


## Loader Utilities


In [ ]:
def _is_scalar(v: Any) -> bool:
    return isinstance(v, (int, float, str, bool)) or v is None


def _to_float(v: Any) -> float | None:
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def _normalize_algorithm_label(raw_algorithm: Any, exp_name: Any) -> str:
    raw = str(raw_algorithm or "").strip().lower()
    exp = str(exp_name or "").strip().lower()

    canonical = [
        "erm",
        "groupdro",
        "iro",
        "inftask",
        "irm",
        "vrex",
        "arm",
        "coral",
        "eqrm",
        "iid",
        "mldg",
    ]

    for name in canonical:
        if raw == name or raw.startswith(name + "(") or (name + "(") in raw:
            return "iro" if name == "inftask" else name

    for name in canonical:
        if exp.endswith(f"_{name}") or f"_{name}_" in exp:
            return "iro" if name == "inftask" else name

    if "(" in raw:
        head = raw.split("(", 1)[0].strip()
        if head in canonical:
            return "iro" if head == "inftask" else head

    return raw if raw else "unknown"


def iter_jsonl_paths(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(root.rglob("*.jsonl"))


def load_iwildcam_records(results_root: Path, dataset_filter: str | None = "iwildcam"):
    run_rows: list[dict[str, Any]] = []
    history_rows: list[dict[str, Any]] = []
    split_rows: list[dict[str, Any]] = []
    eval_rows: list[dict[str, Any]] = []
    error_rows: list[dict[str, Any]] = []

    for path in iter_jsonl_paths(results_root):
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    error_rows.append({
                        "file": str(path),
                        "line": line_no,
                        "error_type": "JSONDecodeError",
                        "error_message": "Could not parse JSON line",
                    })
                    continue

                dataset_name = rec.get("dataset_name")
                if dataset_filter is not None and dataset_name != dataset_filter:
                    continue

                result = rec.get("result") or {}
                raw_algorithm = result.get("algorithm_name") or rec.get("algorithm")
                norm_algorithm = _normalize_algorithm_label(raw_algorithm, rec.get("exp_name"))

                run = {
                    "file": str(path),
                    "line": line_no,
                    "run_id": rec.get("run_id"),
                    "status": rec.get("status"),
                    "experiment": rec.get("experiment"),
                    "exp_name": rec.get("exp_name"),
                    "dataset_name": dataset_name,
                    "source": rec.get("source"),
                    "algorithm": norm_algorithm,
                    "algorithm_raw": str(raw_algorithm) if raw_algorithm is not None else None,
                    "seed": rec.get("seed"),
                    "args_id": rec.get("args_id"),
                    "output_root": rec.get("output_root"),
                }

                for k, v in rec.items():
                    if k in run:
                        continue
                    if isinstance(k, str) and k.endswith(("_acc_final", "_acc_best", "_loss_final", "_loss_best")):
                        run[k] = _to_float(v)

                for k, v in result.items():
                    if _is_scalar(v):
                        run[k] = v

                run_rows.append(run)

                err = rec.get("error")
                if isinstance(err, dict):
                    error_rows.append({
                        "file": str(path),
                        "line": line_no,
                        "run_id": rec.get("run_id"),
                        "algorithm": run.get("algorithm"),
                        "seed": run.get("seed"),
                        "error_type": err.get("type"),
                        "error_message": err.get("message"),
                    })

                history = result.get("history")
                if isinstance(history, list):
                    for event in history:
                        if not isinstance(event, dict):
                            continue
                        step = event.get("step")
                        train = event.get("train") if isinstance(event.get("train"), dict) else {}
                        eval_dict = event.get("eval") if isinstance(event.get("eval"), dict) else {}
                        for split, metric in eval_dict.items():
                            metric = metric if isinstance(metric, dict) else {}
                            history_rows.append({
                                "run_id": run.get("run_id"),
                                "algorithm": run.get("algorithm"),
                                "seed": run.get("seed"),
                                "exp_name": run.get("exp_name"),
                                "step": step,
                                "split": split,
                                "train_loss": _to_float(train.get("loss")),
                                "accuracy": _to_float(metric.get("accuracy")),
                                "macro_f1": _to_float(metric.get("macro_f1")),
                                "macro_recall": _to_float(metric.get("macro_recall")),
                            })

                for stage_key in ("iwildcam_metrics_best", "iwildcam_metrics_final"):
                    stage_payload = result.get(stage_key)
                    if not isinstance(stage_payload, dict):
                        continue
                    stage = stage_key.replace("iwildcam_metrics_", "")
                    for split, metric in stage_payload.items():
                        metric = metric if isinstance(metric, dict) else {}
                        split_rows.append({
                            "run_id": run.get("run_id"),
                            "algorithm": run.get("algorithm"),
                            "seed": run.get("seed"),
                            "exp_name": run.get("exp_name"),
                            "stage": stage,
                            "split": split,
                            "accuracy": _to_float(metric.get("accuracy")),
                            "macro_f1": _to_float(metric.get("macro_f1")),
                            "macro_recall": _to_float(metric.get("macro_recall")),
                            "alpha": _to_float(metric.get("alpha")),
                        })

                eval_metrics = result.get("metrics")
                if isinstance(eval_metrics, list):
                    eval_alpha = _to_float(result.get("eval_alpha"))
                    for metric in eval_metrics:
                        if not isinstance(metric, dict):
                            continue
                        eval_rows.append({
                            "run_id": run.get("run_id"),
                            "algorithm": run.get("algorithm"),
                            "seed": run.get("seed"),
                            "exp_name": run.get("exp_name"),
                            "split": metric.get("split"),
                            "accuracy": _to_float(metric.get("accuracy")),
                            "macro_f1": _to_float(metric.get("macro_f1")),
                            "macro_recall": _to_float(metric.get("macro_recall")),
                            "eval_alpha": eval_alpha if eval_alpha is not None else _to_float(metric.get("alpha")),
                        })

    runs_df = pd.DataFrame(run_rows)
    history_df = pd.DataFrame(history_rows)
    split_df = pd.DataFrame(split_rows)
    eval_df = pd.DataFrame(eval_rows)
    errors_df = pd.DataFrame(error_rows)
    return runs_df, history_df, split_df, eval_df, errors_df


In [ ]:
# Load all records for iWildCam
runs_df, history_df, split_df, eval_df, errors_df = load_iwildcam_records(RESULTS_ROOT, DATASET_FILTER)

print(f"runs rows: {len(runs_df):,}")
print(f"history rows: {len(history_df):,}")
print(f"split metric rows: {len(split_df):,}")
print(f"eval metric rows: {len(eval_df):,}")
print(f"error rows: {len(errors_df):,}")

if not runs_df.empty:
    display(runs_df.head(3))
if not errors_df.empty:
    print("Sample errors:")
    display(errors_df.head(5))


## Run Health and Coverage


In [ ]:
if runs_df.empty:
    print("No iWildCam records found under RESULTS_ROOT.")
else:
    health = (
        runs_df
        .groupby(["algorithm", "status"], dropna=False)
        .size()
        .rename("n")
        .reset_index()
        .sort_values(["algorithm", "status"])
    )
    display(health)

    coverage = (
        runs_df
        .pivot_table(index="algorithm", columns="seed", values="status", aggfunc="first")
        .sort_index()
    )
    display(coverage)


## Build Analysis View (Best-Val Metrics)

This merges either flat run columns (`val_acc_best`, `val_macro_f1_best`, ...) or the structured `iwildcam_metrics_best` payload.


In [ ]:
def build_best_val_view(runs_df: pd.DataFrame, split_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["run_id", "algorithm", "seed", "exp_name", "status"]
    base = runs_df[base_cols].copy() if set(base_cols).issubset(runs_df.columns) else pd.DataFrame(columns=base_cols)

    for col in ["val_acc_best", "val_macro_f1_best", "val_macro_recall_best"]:
        if col in runs_df.columns:
            base[col] = pd.to_numeric(runs_df[col], errors="coerce")

    needs_merge = any(col not in base.columns for col in ["val_acc_best", "val_macro_f1_best", "val_macro_recall_best"])
    if needs_merge and not split_df.empty:
        best_val = split_df[(split_df["stage"] == "best") & (split_df["split"] == "val")].copy()
        best_val = best_val[["run_id", "accuracy", "macro_f1", "macro_recall"]].rename(
            columns={
                "accuracy": "val_acc_best",
                "macro_f1": "val_macro_f1_best",
                "macro_recall": "val_macro_recall_best",
            }
        )
        base = base.merge(best_val, on="run_id", how="left", suffixes=("", "_from_split"))
        for col in ["val_acc_best", "val_macro_f1_best", "val_macro_recall_best"]:
            alt = f"{col}_from_split"
            if alt in base.columns:
                base[col] = base[col].where(base[col].notna(), base[alt]) if col in base.columns else base[alt]
                base = base.drop(columns=[alt])

    return base


best_view = build_best_val_view(runs_df, split_df)
best_ok = best_view[best_view["status"] == "ok"].copy()

print(f"best_view rows: {len(best_view):,}")
print(f"best_ok rows: {len(best_ok):,}")

if not best_ok.empty:
    display(best_ok.sort_values(["algorithm", "seed"]).head(20))


## Algorithm Summary Table


In [ ]:
if best_ok.empty:
    print("No successful runs available for summary.")
else:
    summary = (
        best_ok
        .groupby("algorithm", dropna=False)
        .agg(
            n_runs=("run_id", "count"),
            n_seeds=("seed", "nunique"),
            val_acc_best_mean=("val_acc_best", "mean"),
            val_acc_best_std=("val_acc_best", "std"),
            val_f1_best_mean=("val_macro_f1_best", "mean"),
            val_f1_best_std=("val_macro_f1_best", "std"),
            val_recall_best_mean=("val_macro_recall_best", "mean"),
            val_recall_best_std=("val_macro_recall_best", "std"),
        )
        .reset_index()
        .sort_values("val_acc_best_mean", ascending=False)
    )
    display(summary)


## Visualization: Best Validation Metrics by Algorithm


In [ ]:
if best_ok.empty:
    print("No successful runs available for plotting.")
else:
    plot_df = best_ok.copy()
    plot_df = plot_df[plot_df["algorithm"].notna()].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    metric_specs = [
        ("val_acc_best", "Validation Accuracy (best)", axes[0]),
        ("val_macro_f1_best", "Validation Macro-F1 (best)", axes[1]),
    ]

    order = (
        plot_df.groupby("algorithm")["val_acc_best"]
        .mean()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    for metric, title, ax in metric_specs:
        data = plot_df[["algorithm", "seed", metric]].dropna().copy()
        if data.empty:
            ax.set_title(f"{title} (no data)")
            continue

        stats = (
            data.groupby("algorithm", as_index=False)[metric]
            .agg(mean="mean", std="std")
            .set_index("algorithm")
            .reindex(order)
            .reset_index()
        )
        stats["std"] = stats["std"].fillna(0.0)

        palette = sns.color_palette("Set2", n_colors=len(stats))
        ax.bar(stats["algorithm"], stats["mean"], yerr=stats["std"], capsize=6, color=palette, alpha=0.9)

        sns.stripplot(
            data=data,
            x="algorithm",
            y=metric,
            order=order,
            color="black",
            alpha=0.45,
            size=4,
            jitter=0.15,
            ax=ax,
        )

        ax.set_title(title)
        ax.set_xlabel("algorithm")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=0.25)

    fig.savefig(FIG_DIR / "best_val_metrics_by_algorithm.png", dpi=150, bbox_inches="tight")
    plt.show()


## Visualization: Distribution Across Seeds


In [ ]:
if best_ok.empty:
    print("No successful runs available for box plots.")
else:
    melted = best_ok.melt(
        id_vars=["algorithm", "seed"],
        value_vars=["val_acc_best", "val_macro_f1_best"],
        var_name="metric",
        value_name="value",
    ).dropna()

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=melted, x="algorithm", y="value", hue="metric", palette="tab10")
    sns.stripplot(data=melted, x="algorithm", y="value", hue="metric", dodge=True, alpha=0.45, color="black", size=3)

    handles, labels = plt.gca().get_legend_handles_labels()
    if len(labels) >= 2:
        plt.legend(handles[:2], labels[:2], title="metric", loc="best")

    plt.title("Seed-Level Metric Distributions")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "seed_distributions_boxplot.png", dpi=150, bbox_inches="tight")
    plt.show()


## Training Dynamics from `result.history`


In [ ]:
if history_df.empty:
    print("No history rows found. This can happen if runs failed or used eval-only records.")
else:
    val_hist = history_df[history_df["split"] == "val"].copy()
    if val_hist.empty:
        print("No 'val' split rows found in history.")
    else:
        val_hist["step"] = pd.to_numeric(val_hist["step"], errors="coerce")
        val_hist = val_hist.dropna(subset=["step"]).sort_values(["algorithm", "seed", "step"])

        agg = (
            val_hist
            .groupby(["algorithm", "step"], as_index=False)
            .agg(
                acc_mean=("accuracy", "mean"),
                acc_std=("accuracy", "std"),
                f1_mean=("macro_f1", "mean"),
                f1_std=("macro_f1", "std"),
            )
        )
        agg[["acc_std", "f1_std"]] = agg[["acc_std", "f1_std"]].fillna(0.0)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)
        palette = sns.color_palette("tab10", n_colors=max(1, agg["algorithm"].nunique()))
        color_map = {alg: palette[i % len(palette)] for i, alg in enumerate(sorted(agg["algorithm"].dropna().unique()))}

        for alg in sorted(agg["algorithm"].dropna().unique()):
            g = agg[agg["algorithm"] == alg].sort_values("step")
            c = color_map[alg]

            axes[0].plot(g["step"], g["acc_mean"], label=alg, color=c, linewidth=2.5)
            axes[0].fill_between(g["step"], g["acc_mean"] - g["acc_std"], g["acc_mean"] + g["acc_std"], alpha=0.2, color=c)

            axes[1].plot(g["step"], g["f1_mean"], label=alg, color=c, linewidth=2.5)
            axes[1].fill_between(g["step"], g["f1_mean"] - g["f1_std"], g["f1_mean"] + g["f1_std"], alpha=0.2, color=c)

        axes[0].set_title("Validation Accuracy vs Step (mean ± std)")
        axes[1].set_title("Validation Macro-F1 vs Step (mean ± std)")
        axes[0].set_xlabel("step"); axes[1].set_xlabel("step")
        axes[0].set_ylabel("accuracy"); axes[1].set_ylabel("macro_f1")
        axes[0].legend(); axes[1].legend()

        plt.tight_layout()
        fig.savefig(FIG_DIR / "val_training_dynamics.png", dpi=150, bbox_inches="tight")
        plt.show()


## Optional: IRO Alpha Sweep (eval mode) + GroupDRO Comparison

The earlier `eval_df` table is empty until dedicated `iro eval` runs are created.

This section can generate those records directly from Python using `run_evaluation(...)`, cache them, and then produce a direct comparison between:
- IRO as a function of `eval.alpha`
- GroupDRO as an alpha-invariant baseline


In [ ]:
# Optional: run eval-mode alpha sweep for IRO checkpoints and cache results
from iro import run_evaluation

ALPHA_GRID = np.round(np.linspace(0.0, 1.0, 11), 2)
EVAL_SPLIT_FOR_SWEEP = "val"
RUN_IRO_ALPHA_SWEEP = False   # set True to launch eval jobs from notebook
MAX_IRO_RUNS = None           # set integer to limit number of IRO checkpoints

EVAL_DATA_ROOT = Path("/home/c01josh/CISPA-scratch/c01josh/datasets/iwildcam")
EVAL_OUTPUT_ROOT = Path("/home/c01josh/CISPA-scratch/c01josh/iro/runs/iwildcam_alpha_eval")
EVAL_EXP_NAME = "iwildcam_alpha_eval"
ALPHA_CACHE_CSV = OUT_DIR / "iro_alpha_eval_cache.csv"


def checkpoint_path_from_row(row: pd.Series, which: str = "best") -> Path:
    output_root = Path(str(row.get("output_root") or "/home/c01josh/CISPA-scratch/c01josh/iro/iro_exp"))
    return output_root / "ckpts" / f"{row['run_id']}_{which}.pkl"


iro_runs = best_ok[(best_ok["algorithm"] == "iro") & best_ok["run_id"].notna()].copy()
if MAX_IRO_RUNS is not None and len(iro_runs):
    iro_runs = iro_runs.sort_values("val_acc_best", ascending=False).head(int(MAX_IRO_RUNS))

if len(iro_runs):
    iro_runs["ckpt_best"] = iro_runs.apply(checkpoint_path_from_row, axis=1)
    iro_runs["ckpt_exists"] = iro_runs["ckpt_best"].map(lambda p: p.exists())

display_cols = ["run_id", "seed", "exp_name", "val_acc_best", "val_macro_f1_best", "ckpt_best", "ckpt_exists"]
display(iro_runs[display_cols] if len(iro_runs) else iro_runs)

alpha_eval_rows: list[dict[str, Any]] = []

if RUN_IRO_ALPHA_SWEEP:
    if iro_runs.empty:
        raise RuntimeError("No successful IRO runs found to evaluate.")

    for _, row in iro_runs.iterrows():
        ckpt = Path(row["ckpt_best"])
        if not ckpt.exists():
            print(f"Skipping missing checkpoint: {ckpt}")
            continue

        for alpha in ALPHA_GRID:
            overrides = [
                f"data.root={EVAL_DATA_ROOT}",
                f"data.root_dir={EVAL_DATA_ROOT}",
                f"data.data_dir={EVAL_DATA_ROOT}",
                "data.download=false",
                "iro.algorithm=iro",
                f"training.seed={int(row['seed']) if pd.notna(row['seed']) else 0}",
                "training.device=auto",
                f"training.output_root={EVAL_OUTPUT_ROOT}",
                f"training.exp_name={EVAL_EXP_NAME}",
                f"eval.checkpoint_path={ckpt}",
                f"eval.split={EVAL_SPLIT_FOR_SWEEP}",
                f"eval.alpha={float(alpha):.4f}",
            ]
            try:
                result = run_evaluation(experiment="iwildcam_iro", overrides=overrides)
                metric = (result.get("metrics_by_split") or {}).get(EVAL_SPLIT_FOR_SWEEP, {})
                alpha_eval_rows.append({
                    "run_id": row["run_id"],
                    "seed": row["seed"],
                    "exp_name": row["exp_name"],
                    "eval_alpha": float(alpha),
                    "split": EVAL_SPLIT_FOR_SWEEP,
                    "acc": _to_float(metric.get("accuracy")),
                    "macro_f1": _to_float(metric.get("macro_f1")),
                    "macro_recall": _to_float(metric.get("macro_recall")),
                    "checkpoint_path": str(ckpt),
                })
            except Exception as exc:
                alpha_eval_rows.append({
                    "run_id": row["run_id"],
                    "seed": row["seed"],
                    "exp_name": row["exp_name"],
                    "eval_alpha": float(alpha),
                    "split": EVAL_SPLIT_FOR_SWEEP,
                    "acc": np.nan,
                    "macro_f1": np.nan,
                    "macro_recall": np.nan,
                    "checkpoint_path": str(ckpt),
                    "error": str(exc),
                })
                print(f"Eval failed for run_id={row['run_id']} alpha={alpha}: {exc}")

    alpha_eval_df = pd.DataFrame(alpha_eval_rows)
    if not alpha_eval_df.empty:
        alpha_eval_df.to_csv(ALPHA_CACHE_CSV, index=False)
        print(f"Saved alpha sweep cache: {ALPHA_CACHE_CSV}")
else:
    if ALPHA_CACHE_CSV.exists():
        alpha_eval_df = pd.read_csv(ALPHA_CACHE_CSV)
        print(f"Loaded cached alpha sweep: {ALPHA_CACHE_CSV}")
    else:
        alpha_eval_df = eval_df[(eval_df["algorithm"] == "iro") & (eval_df["split"] == EVAL_SPLIT_FOR_SWEEP)].copy()
        if not alpha_eval_df.empty:
            alpha_eval_df = alpha_eval_df.rename(columns={"accuracy": "acc"})
            print("Using existing eval_df rows for IRO alpha sweep (no fresh eval run).")
        else:
            alpha_eval_df = pd.DataFrame(columns=["run_id", "seed", "eval_alpha", "split", "acc", "macro_f1", "macro_recall", "checkpoint_path"])
            print("No eval-mode metrics detected yet. Set RUN_IRO_ALPHA_SWEEP=True to generate them.")

if not alpha_eval_df.empty:
    alpha_eval_df["eval_alpha"] = pd.to_numeric(alpha_eval_df["eval_alpha"], errors="coerce")
    alpha_eval_df["acc"] = pd.to_numeric(alpha_eval_df["acc"], errors="coerce")
    alpha_eval_df["macro_f1"] = pd.to_numeric(alpha_eval_df["macro_f1"], errors="coerce")
    alpha_eval_df["macro_recall"] = pd.to_numeric(alpha_eval_df["macro_recall"], errors="coerce")

display(alpha_eval_df.head(10))


In [ ]:
# Plot IRO alpha-sweep curve against GroupDRO baseline
if alpha_eval_df.empty:
    print("No alpha-eval rows available. Run the previous cell with RUN_IRO_ALPHA_SWEEP=True.")
else:
    d = alpha_eval_df.dropna(subset=["eval_alpha", "acc", "macro_f1"]).copy()
    d = d[d["split"] == EVAL_SPLIT_FOR_SWEEP]

    if d.empty:
        print("Alpha sweep table exists but has no valid val metrics.")
    else:
        curve = (
            d.groupby("eval_alpha", as_index=False)
             .agg(
                 acc_mean=("acc", "mean"),
                 acc_std=("acc", "std"),
                 f1_mean=("macro_f1", "mean"),
                 f1_std=("macro_f1", "std"),
             )
             .sort_values("eval_alpha")
        )
        curve[["acc_std", "f1_std"]] = curve[["acc_std", "f1_std"]].fillna(0.0)
        display(curve)

        gd = best_ok[best_ok["algorithm"] == "groupdro"].copy()
        erm = best_ok[best_ok["algorithm"] == "erm"].copy()

        gd_acc_mean = gd["val_acc_best"].mean() if not gd.empty else np.nan
        gd_acc_std = gd["val_acc_best"].std(ddof=1) if len(gd) > 1 else 0.0
        gd_f1_mean = gd["val_macro_f1_best"].mean() if not gd.empty else np.nan
        gd_f1_std = gd["val_macro_f1_best"].std(ddof=1) if len(gd) > 1 else 0.0

        erm_acc_mean = erm["val_acc_best"].mean() if not erm.empty else np.nan
        erm_f1_mean = erm["val_macro_f1_best"].mean() if not erm.empty else np.nan

        x_min = float(curve["eval_alpha"].min())
        x_max = float(curve["eval_alpha"].max())

        fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

        # Accuracy panel
        axes[0].plot(curve["eval_alpha"], curve["acc_mean"], marker="o", linewidth=2.5, color="#1f77b4", label="IRO eval(alpha)")
        axes[0].fill_between(curve["eval_alpha"], curve["acc_mean"] - curve["acc_std"], curve["acc_mean"] + curve["acc_std"], color="#1f77b4", alpha=0.2)

        if np.isfinite(gd_acc_mean):
            axes[0].hlines(gd_acc_mean, x_min, x_max, colors="#d62728", linestyles="--", linewidth=2.2, label="GroupDRO (mean)")
            axes[0].fill_between([x_min, x_max], gd_acc_mean - gd_acc_std, gd_acc_mean + gd_acc_std, color="#d62728", alpha=0.14)

        if np.isfinite(erm_acc_mean):
            axes[0].hlines(erm_acc_mean, x_min, x_max, colors="#2ca02c", linestyles=":", linewidth=2.0, label="ERM (mean)")

        axes[0].set_title("Validation Accuracy")
        axes[0].set_xlabel("eval.alpha")
        axes[0].set_ylabel("accuracy")
        axes[0].legend()
        axes[0].grid(alpha=0.25)

        # Macro-F1 panel
        axes[1].plot(curve["eval_alpha"], curve["f1_mean"], marker="s", linewidth=2.5, color="#1f77b4", label="IRO eval(alpha)")
        axes[1].fill_between(curve["eval_alpha"], curve["f1_mean"] - curve["f1_std"], curve["f1_mean"] + curve["f1_std"], color="#1f77b4", alpha=0.2)

        if np.isfinite(gd_f1_mean):
            axes[1].hlines(gd_f1_mean, x_min, x_max, colors="#d62728", linestyles="--", linewidth=2.2, label="GroupDRO (mean)")
            axes[1].fill_between([x_min, x_max], gd_f1_mean - gd_f1_std, gd_f1_mean + gd_f1_std, color="#d62728", alpha=0.14)

        if np.isfinite(erm_f1_mean):
            axes[1].hlines(erm_f1_mean, x_min, x_max, colors="#2ca02c", linestyles=":", linewidth=2.0, label="ERM (mean)")

        axes[1].set_title("Validation Macro-F1")
        axes[1].set_xlabel("eval.alpha")
        axes[1].set_ylabel("macro_f1")
        axes[1].legend()
        axes[1].grid(alpha=0.25)

        fig.savefig(FIG_DIR / "iro_alpha_vs_groupdro_val.png", dpi=150, bbox_inches="tight")
        plt.show()


## Optional: DGIL-Style Risk Distribution on Training Groups

This section approximates the DGIL simulation-style risk distribution plot for iWildCam by:
- taking one trained IRO checkpoint
- evaluating train-split per-example cross-entropy
- aggregating by environment group (`location`)
- plotting KDE curves of group-risk distributions for multiple `alpha` values

Notes:
- this can be expensive on full train data
- use `RISK_MAX_BATCHES` for a faster first pass


In [ ]:
# Optional: DGIL-style risk distribution over train-group risks
import torch
import torch.nn.functional as F

from wilds import get_dataset
from wilds.common.data_loaders import get_eval_loader
from wilds.common.grouper import CombinatorialGrouper

from iro.core.config import load_experiment_config
from iro.data.iwildcam import iwildcam_eval_transform
from iro.training.train_iwildcam import _build_algorithm

RUN_RISK_DISTRIBUTION = False  # set True to execute
RISK_ALPHAS = np.round(np.linspace(0.0, 0.9, 10), 2)
RISK_BATCH_SIZE = 64
RISK_NUM_WORKERS = 4
RISK_MAX_BATCHES = None        # e.g. 200 for a quick pass
RISK_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RISK_DATA_ROOT = EVAL_DATA_ROOT
RISK_CACHE_CSV = OUT_DIR / "iro_train_group_risk_distribution.csv"
RISK_FIG_PATH = FIG_DIR / "iro_train_group_risk_kde.png"


def _cvar(values: np.ndarray, alpha: float) -> float:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.nan
    q = np.quantile(values, alpha)
    tail = values[values >= q]
    return float(tail.mean()) if tail.size else float(q)


def _pick_reference_iro_checkpoint(iro_runs_df: pd.DataFrame) -> Path | None:
    if iro_runs_df.empty:
        return None
    tmp = iro_runs_df.copy()
    if "ckpt_exists" in tmp.columns:
        tmp = tmp[tmp["ckpt_exists"] == True]
    if tmp.empty:
        return None
    if "val_acc_best" in tmp.columns:
        tmp = tmp.sort_values("val_acc_best", ascending=False)
    return Path(tmp.iloc[0]["ckpt_best"])


risk_summary_df = pd.DataFrame()

if RUN_RISK_DISTRIBUTION:
    ckpt = _pick_reference_iro_checkpoint(iro_runs)
    if ckpt is None or not ckpt.exists():
        raise RuntimeError("No valid IRO checkpoint available for risk distribution plotting.")

    print(f"Using checkpoint: {ckpt}")

    cfg = load_experiment_config(
        experiment="iwildcam_iro",
        config_root=str(Path.cwd() / "config"),
        overrides=[
            f"data.root={RISK_DATA_ROOT}",
            f"data.root_dir={RISK_DATA_ROOT}",
            f"data.data_dir={RISK_DATA_ROOT}",
            "data.download=false",
            "iro.algorithm=iro",
            "model.name=film_resnet18",
            f"data.batch_size={RISK_BATCH_SIZE}",
            f"data.num_workers={RISK_NUM_WORKERS}",
        ],
    )

    dataset = get_dataset(dataset="iwildcam", root_dir=str(RISK_DATA_ROOT), download=False)
    train_subset = dataset.get_subset("train", transform=iwildcam_eval_transform())
    train_loader = get_eval_loader(
        "standard",
        train_subset,
        batch_size=RISK_BATCH_SIZE,
        num_workers=RISK_NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    grouper = CombinatorialGrouper(dataset=dataset, groupby_fields=["location"])

    algorithm, _hparams = _build_algorithm(cfg, n_classes=int(dataset.n_classes), steps=1)
    payload = torch.load(ckpt, map_location=RISK_DEVICE)
    state = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
    algorithm.load_state_dict(state, strict=False)
    algorithm.to(RISK_DEVICE)
    algorithm.eval()

    alpha_to_group_risks: dict[float, np.ndarray] = {}
    summary_rows: list[dict[str, Any]] = []

    for alpha in RISK_ALPHAS:
        group_loss_sum: dict[int, float] = {}
        group_count: dict[int, int] = {}

        with torch.no_grad():
            for batch_idx, (x, y, metadata) in enumerate(train_loader):
                if RISK_MAX_BATCHES is not None and batch_idx >= int(RISK_MAX_BATCHES):
                    break

                x = x.to(RISK_DEVICE)
                y = y.to(RISK_DEVICE).view(-1).long()

                alpha_t = torch.tensor(float(alpha), device=x.device, dtype=x.dtype)
                logits = algorithm.predict(x, alpha_t)
                losses = F.cross_entropy(logits, y, reduction="none").detach().cpu()

                group_ids = grouper.metadata_to_group(metadata.detach().cpu())
                for gid in torch.unique(group_ids):
                    g = int(gid.item())
                    mask = group_ids == gid
                    group_loss_sum[g] = group_loss_sum.get(g, 0.0) + float(losses[mask].sum().item())
                    group_count[g] = group_count.get(g, 0) + int(mask.sum().item())

        risks = np.array([group_loss_sum[g] / max(group_count[g], 1) for g in sorted(group_loss_sum)], dtype=float)
        alpha_to_group_risks[float(alpha)] = risks

        summary_rows.append({
            "alpha": float(alpha),
            "n_groups": int(risks.size),
            "risk_mean": float(np.mean(risks)) if risks.size else np.nan,
            "risk_std": float(np.std(risks, ddof=1)) if risks.size > 1 else 0.0,
            "risk_cvar": _cvar(risks, float(alpha)),
        })

    risk_summary_df = pd.DataFrame(summary_rows)
    risk_summary_df.to_csv(RISK_CACHE_CSV, index=False)
    display(risk_summary_df)

    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    colors = sns.color_palette("RdBu_r", n_colors=len(RISK_ALPHAS))

    for color, alpha in zip(colors, RISK_ALPHAS):
        risks = alpha_to_group_risks.get(float(alpha), np.array([], dtype=float))
        if risks.size == 0:
            continue
        if risks.size > 2 and float(np.std(risks)) > 1e-9:
            sns.kdeplot(risks, ax=ax, color=color, linewidth=2.0)
        else:
            ax.axvline(float(np.mean(risks)), color=color, linewidth=2.0)

    ax.set_title("Distribution of Training-Group Risks by Preference Alpha")
    ax.set_xlabel("group risk (mean cross-entropy)")
    ax.set_ylabel("density")

    norm = plt.Normalize(float(np.min(RISK_ALPHAS)), float(np.max(RISK_ALPHAS)))
    sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"$\alpha$", labelpad=5, size=12)

    plt.tight_layout()
    fig.savefig(RISK_FIG_PATH, dpi=150, bbox_inches="tight")
    plt.show()
else:
    if RISK_CACHE_CSV.exists():
        risk_summary_df = pd.read_csv(RISK_CACHE_CSV)
        print(f"Loaded cached risk summary: {RISK_CACHE_CSV}")
        display(risk_summary_df)

        plt.figure(figsize=(8, 5))
        plt.plot(risk_summary_df["alpha"], risk_summary_df["risk_cvar"], marker="o", linewidth=2.2)
        plt.title("Cached Train-Group CVaR by Alpha")
        plt.xlabel("alpha")
        plt.ylabel("CVaR of group risks")
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "iro_train_group_cvar_vs_alpha.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Risk distribution cache not found. Set RUN_RISK_DISTRIBUTION=True to generate it.")


## Export Tables


In [ ]:
runs_csv = OUT_DIR / "iwildcam_runs_flat.csv"
history_csv = OUT_DIR / "iwildcam_history_flat.csv"
split_csv = OUT_DIR / "iwildcam_split_metrics_flat.csv"
eval_csv = OUT_DIR / "iwildcam_eval_metrics_flat.csv"
errors_csv = OUT_DIR / "iwildcam_errors_flat.csv"

runs_df.to_csv(runs_csv, index=False)
history_df.to_csv(history_csv, index=False)
split_df.to_csv(split_csv, index=False)
eval_df.to_csv(eval_csv, index=False)
errors_df.to_csv(errors_csv, index=False)

if "alpha_eval_df" in globals() and isinstance(alpha_eval_df, pd.DataFrame) and not alpha_eval_df.empty:
    alpha_eval_csv = OUT_DIR / "iro_alpha_eval_metrics.csv"
    alpha_eval_df.to_csv(alpha_eval_csv, index=False)
else:
    alpha_eval_csv = None

if "risk_summary_df" in globals() and isinstance(risk_summary_df, pd.DataFrame) and not risk_summary_df.empty:
    risk_summary_csv = OUT_DIR / "iro_train_group_risk_summary.csv"
    risk_summary_df.to_csv(risk_summary_csv, index=False)
else:
    risk_summary_csv = None

print("Wrote:")
print(" -", runs_csv)
print(" -", history_csv)
print(" -", split_csv)
print(" -", eval_csv)
print(" -", errors_csv)
if alpha_eval_csv is not None:
    print(" -", alpha_eval_csv)
if risk_summary_csv is not None:
    print(" -", risk_summary_csv)
print("Figure directory:", FIG_DIR)


## Notes

- For model-selection comparisons, use `*_best` metrics.
- For "what the final checkpoint does", use `*_final` metrics.
- iWildCam metrics can be low early in training; compare across equal step budgets.
- For preference analysis, sweep `eval.alpha` on fixed IRO checkpoints and re-run this notebook.
